# 04 — Modules, Imports & Project Structure
### How 8 files become one program

This is the notebook that explains *why* the project is split into
`models.py`, `database.py`, `escalation_rules.py`, `retrieval.py`,
`llm_client.py`, `agents.py`, `pipeline.py`, and `main.py` instead of one
giant script — and exactly how Python wires them together via `import`.

## 4.1 A module is just a file

Any `.py` file is a **module**. When you write `import database`, Python
runs `database.py` top to bottom once, then gives you a `database` object
whose attributes are everything defined at that file's top level —
functions, classes, constants.

In [ ]:
import sys
sys.path.insert(0, ".")   # ensure the current directory is importable

import database
print(type(database))
print(database.DB_PATH)          # a module-level constant defined in database.py
print(database.get_customer)     # a function defined in database.py, accessed as an attribute


## 4.2 `from module import name`

If you only need a few specific things from a module, `from ... import ...`
pulls them directly into your namespace, so you don't have to prefix every
use with the module name.

In [ ]:
from database import get_customer, init_db

init_db(force=True)
print(get_customer("CUST001"))


## 4.3 Why the real project imports this way

`agents.py` does exactly this:

```python
import database
import llm_client
from retrieval import KnowledgeRetriever
```

Notice the mix: `database` and `llm_client` are imported as whole modules
(because `agents.py` calls several different functions from each —
`database.get_customer`, `database.get_orders`,
`database.get_prior_ticket_count`), while only the one class it actually
needs, `KnowledgeRetriever`, is imported by name from `retrieval`. That's a
readability convention, not a hard rule — import the module when you use
many things from it, import specific names when you only need one or two.

## 4.4 The dependency chain

Each file only imports what it actually needs, and nothing imports in a
circle. Here's the real chain, bottom to top:

```
models.py            <- no project imports (just pydantic)
database.py          <- no project imports (just sqlite3, pathlib)
escalation_rules.py  <- no project imports (pure logic)
retrieval.py          <- no project imports (just sklearn)
llm_client.py         <- no project imports (just os, re, json)
                         |
agents.py  -----------+  imports database, llm_client, retrieval
                         |
pipeline.py  ---------+  imports agents, escalation_rules
                         |
main.py, test_tickets.py  import pipeline (and database, for --init-db)
```

This shape matters: `database.py` has zero idea that `agents.py` exists, and
that's exactly right — a low-level module (talks to SQLite) should never
need to know about the high-level code that uses it. If you ever catch
yourself wanting `database.py` to import something from `pipeline.py`,
that's a sign the two files' responsibilities have gotten tangled.

In [ ]:
# You can inspect this at runtime, too: every module has a __name__ and,
# once imported, shows up in sys.modules.
import agents
print("agents" in sys.modules)
print("database" in sys.modules)   # already True -- agents.py imported it, which ran it


## 4.5 The `if __name__ == "__main__":` guard

Every file in the project ends with something like:

```python
if __name__ == "__main__":
    init_db(force=True)
    print(f"Seeded database at {DB_PATH}")
```

`__name__` is a special variable every module has. It equals `"__main__"`
**only** when that file is the one you ran directly (`python3 database.py`)
— when the same file is instead *imported* by another module, `__name__`
is set to the module's own name (`"database"`) instead. This is what lets
`database.py` double as both:

- a library other files import from (`from database import get_customer`)
- a standalone script you can run directly to seed the DB and see a
  confirmation message

...without the "seed and print" behavior accidentally firing every time
some other file imports it.

In [ ]:
print(__name__)   # inside this notebook, this will show the module name Jupyter assigns it -- never "__main__" for a notebook cell like this


## 4.6 Package structure vs. flat modules

This project is a **flat** collection of modules in one folder — no
subfolders, no `__init__.py`. That's a deliberate simplicity choice
appropriate for a project this size. A proper Python *package* (importable
as `import supportpilot` from outside its own folder) would need an
`__init__.py` and be organized as `supportpilot/agents.py`,
`supportpilot/database.py`, etc. — worth knowing the term, but not
something this project needs, since everything runs from inside the one
folder via `main.py`.

## Exercise

1. Create a new file `greetings.py` with one function,
   `greet(name: str) -> str`, that returns `f"Hello, {name}!"`.
2. In a separate cell in this notebook, `import greetings` and call
   `greetings.greet("your name")`.
3. Add an `if __name__ == "__main__":` block to `greetings.py` that calls
   `greet("World")` and prints the result. Run the file directly from a
   terminal (`python3 greetings.py`) and confirm the greeting prints —
   then confirm that importing it from this notebook does *not* print
   anything on its own.
4. Open `main.py` in the project folder and trace, import by import, every
   module it pulls in, directly or indirectly (i.e. what does `pipeline`
   import, and what does *that* import). Draw the dependency chain the way
   Section 4.4 does above, but for `main.py` specifically.